# IntZ Example 9: CSV Query Patterns
**EPS Research IntZ Kinematic Corpus v1.0**

Demonstrate portable query patterns on the corrected flat CSV using only
Python's standard-library `csv` module.


In [1]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'intz_corpus_v1b.json': 'https://zenodo.org/records/21841382/files/intz_corpus_v1b.json',
        'intz_corpus_v1b_flat.csv': 'https://zenodo.org/records/21841382/files/intz_corpus_v1b_flat.csv',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


Running locally — corpus files loaded from working directory.


In [2]:
import csv

CSV_PATH = 'intz_corpus_v1b_flat.csv'

with open(CSV_PATH, newline='') as f:
    rows = list(csv.DictReader(f))

print(f"Loaded {len(rows)} rows, {len(rows[0])} columns")
print(list(rows[0].keys()))


Loaded 1292 rows, 22 columns
['galaxy_id', 'survey', 'quality_tier', 'z_spec', 'z_source', 'ra_deg', 'dec_deg', 'log_mstar', 'sfr_msun_yr', 'Reff_kpc', 'axis_ratio', 'inclination_deg', 'Vc_kms', 'sigma0_kms', 'v_over_sigma', 'kin_type', 'beam_smear_corrected', 'omega_available', 'omega_value_rad_gyr', 'omega_quality', 'omega_outlier_flag', 'reference']


In [3]:
# Filter Tier-1 galaxies
t1 = [r for r in rows if r['quality_tier'] == '1']

print(f"Tier-1: {len(t1)} galaxies")
print("ID                  z       Vc      V/sigma")
print("-"*48)

for r in t1[:10]:
    print(f"{r['galaxy_id']:<18} {r['z_spec']:>7} "
          f"{r['Vc_kms']:>8} {r['v_over_sigma']:>10}")


Tier-1: 166 galaxies
   galaxy_id    z_spec      Vc_kms  omega_value_rad_gyr
0          5  0.849777  129.024292              -5.8754
2         11  0.844478   47.960449              -7.5238
5         15  1.013720   40.536770              -2.9364
9         20  0.833362  151.651627              -8.5005
11        22  0.854378  123.003471             -21.0917
15        26  0.837025   40.933868              -5.7204
16        27  0.783404   65.954117              -4.0373
17        28  0.839406   78.201401              -3.4570
18        29  0.835713   83.728935              -5.7185
26        38  0.838768  117.184723             -24.7077


In [4]:
# High V/sigma galaxies
high_vos = []

for r in rows:
    try:
        if float(r['v_over_sigma']) > 5:
            high_vos.append(r)
    except (ValueError, TypeError):
        pass

print(f"V/sigma > 5: {len(high_vos)} galaxies")
for r in high_vos[:10]:
    print(r['galaxy_id'], r['survey'], r['z_spec'], r['v_over_sigma'])


v/sigma > 5: 101 galaxies
    galaxy_id survey    z_spec  v_over_sigma
3          13  KROSS  0.845449         5.075
4          14  KROSS  0.844147        43.995
24         35  KROSS  0.842692         6.330
41         56  KROSS  0.847649         5.187
43         59  KROSS  0.850111         5.595
56         74  KROSS  0.960410         7.101
71         92  KROSS  0.961797         6.631
81        106  KROSS  0.896229         6.273
82        107  KROSS  0.800873         8.067
101       134  KROSS  0.851508        19.979


In [5]:
# KMOS3D galaxies at z > 2
massive_highz = []

for r in rows:
    try:
        if r['survey'] == 'KMOS3D' and float(r['z_spec']) > 2.0:
            massive_highz.append(r)
    except (ValueError, TypeError):
        pass

print(f"KMOS3D z>2: {len(massive_highz)} galaxies")
for r in massive_highz[:10]:
    print(r['galaxy_id'], r['z_spec'], r['log_mstar'])


KMOS3D z>2: 200 galaxies
